# OpenAI gpt-oss on Amazon Bedrock

> **Sample code — not for production.** Provided as AWS Content under the AWS
> Customer Agreement; do not use it in production accounts or on production or other
> critical data. Running these cells calls Amazon Bedrock and incurs charges. Full
> disclaimer in the [README](../README.md#disclaimer).

OpenAI's open-weight gpt-oss models reason before they answer. `openai.gpt-oss-120b` is
the default here, and `openai.gpt-oss-20b` is smaller and cheaper. The safeguard
variants, which classify content against a policy you write, are in
[`02-gpt-oss-safeguard-classification`](02-gpt-oss-safeguard-classification.ipynb).
OpenAI's hosted GPT models are a different family, in
[`01-openai-gpt/01`](../01-openai-gpt/01-responses-api-core.ipynb).

| Model | `bedrock-mantle` ID | `bedrock-runtime` ID | Notes |
|---|---|---|---|
| gpt-oss 120B | `openai.gpt-oss-120b` | `openai.gpt-oss-120b-1:0` | The default here |
| gpt-oss 20B | `openai.gpt-oss-20b` | `openai.gpt-oss-20b-1:0` | Smaller and cheaper |

IDs as of September 2026, in `us-east-1`. Which Regions carry each model changes as
models launch, so the setup cell checks your Region.

On `bedrock-mantle` these models serve the OpenAI-compatible **Chat Completions** API at
the bare `/v1` path, and the **Responses** API on the same path (section 7). On
`bedrock-runtime` they serve **Converse**, under IDs that end in `-1:0` (section 8).

**Covered:** first call, streaming, multi-turn, reasoning, tool use, structured output,
the Responses API with server-side state, and Converse on `bedrock-runtime`.

## Setup

Install the pinned requirements first (`pip install -r ../requirements.txt`, Python 3.11
or later). The notebook calls Bedrock with the OpenAI SDK and boto3. From
`_shared/bedrock.py`, this repository's own helper module, it takes an availability
check and a lenient JSON parser.

Endpoints, authentication and URL paths are explained in
[`00-foundations/01`](../00-foundations/01-endpoints-auth-and-the-three-paths.ipynb).

In [1]:
import json
import sys

import boto3
from aws_bedrock_token_generator import provide_token
from openai import OpenAI

sys.path.insert(0, "../_shared")
from bedrock import endpoints_for, parse_json_lenient

REGION = "us-east-1"
GPT_OSS_120B = "openai.gpt-oss-120b"
GPT_OSS_20B = "openai.gpt-oss-20b"

# bedrock-mantle: the OpenAI SDK, with a short-term Bedrock API key minted from your
# AWS credentials. Mint it when you build the client, because the key expires.
client = OpenAI(
    base_url=f"https://bedrock-mantle.{REGION}.api.aws/v1",
    api_key=provide_token(region=REGION),
    timeout=120,
)

# bedrock-runtime: boto3 signs each request with SigV4, so there is no key to mint.
runtime = boto3.client("bedrock-runtime", region_name=REGION)

for model in (GPT_OSS_120B, GPT_OSS_20B,):
    served = [name for name, ok in endpoints_for(model, REGION).items() if ok]
    print(f"{model:21} served by: {', '.join(served) or 'neither endpoint in ' + REGION}")

openai.gpt-oss-120b   served by: mantle, runtime
openai.gpt-oss-20b    served by: mantle, runtime


## 1. First call

A single request with a short question. If the model spends the whole `max_tokens`
budget before it answers, `content` is `None` and `finish_reason` is `length`, so the
cell prints both.

In [2]:
response = client.chat.completions.create(
    model=GPT_OSS_120B,
    messages=[{"role": "user", "content": "What is the capital of Australia? Answer in one word."}],
    max_tokens=1000,
)
choice = response.choices[0]
print((choice.message.content or "").strip())
print(f"finish_reason={choice.finish_reason}  total_tokens={response.usage.total_tokens}")

Canberra
finish_reason=stop  total_tokens=103


## 2. Streaming

Set `stream=True` and read `delta.content` from each chunk. Check that a chunk has
choices before indexing it, and read `finish_reason` from the last one: `stop` means the
answer is complete.

In [3]:
stream = client.chat.completions.create(
    model=GPT_OSS_120B,
    messages=[{"role": "user", "content": "Count from 1 to 10, separated by commas."}],
    max_tokens=1000,
    stream=True,
)
finish_reason = None
for chunk in stream:
    if chunk.choices and chunk.choices[0].delta.content:
        print(chunk.choices[0].delta.content, end="", flush=True)
    if chunk.choices and chunk.choices[0].finish_reason:
        finish_reason = chunk.choices[0].finish_reason
print(f"\nfinish_reason={finish_reason}")

1, 2, 3,

 4, 5, 6, 7, 8, 9, 10


finish_reason=stop


## 3. Multi-turn

Chat Completions keeps no conversation state on the server, so each request carries the
whole history. Append the assistant's reply before the next user turn.

In [4]:
messages = [
    {"role": "system", "content": "You are a concise assistant."},
    {"role": "user", "content": "My favourite colour is teal. Reply with one word: noted."},
]
first = client.chat.completions.create(model=GPT_OSS_120B, messages=messages, max_tokens=1000)
messages.append({"role": "assistant", "content": first.choices[0].message.content})
messages.append({"role": "user", "content": "What is my favourite colour? One word."})
second = client.chat.completions.create(model=GPT_OSS_120B, messages=messages, max_tokens=1000)

print("turn 1:", (first.choices[0].message.content or "").strip())
print("turn 2:", (second.choices[0].message.content or "").strip())

turn 1: noted
turn 2: teal


## 4. Reasoning

Set `reasoning_effort` to `low`, `medium` or `high`. The reasoning comes back in
`choices[0].message.reasoning`, a field the OpenAI schema does not define. The SDK still
exposes it as an attribute, but code written against the published types will not see
it. Reasoning is billed as output and spends the budget before the answer starts, so
give it a larger `max_tokens`. The usage figures do not itemise the reasoning:
`completion_tokens_details` is `None`, so budget from `completion_tokens`, which
includes it. In one September 2026 run this question used 22 completion tokens at `low`,
91 at `medium` and 186 at `high`, and the answer was $0.05 each time.

In [5]:
response = client.chat.completions.create(
    model=GPT_OSS_120B,
    messages=[{"role": "user", "content": 'A bat and a ball cost $1.10 in total. The bat costs $1.00 more than the ball. How much does the ball cost? Answer with the amount only.'}],
    reasoning_effort="high",
    max_tokens=4000,
)
message = response.choices[0].message
trace = getattr(message, "reasoning", None) or ""
print("answer   :", (message.content or "").strip())
print(f"reasoning: {len(trace)} characters")
print(f"finish_reason={response.choices[0].finish_reason}  "
      f"completion_tokens={response.usage.completion_tokens}")

answer   : 0.05
reasoning: 473 characters
finish_reason=stop  completion_tokens=185


## 5. Tool use

A tool loop sends the tool definitions, runs each call the model makes, returns the
results, and repeats until the model answers in text. Check each call's arguments before
running it: `finish_reason == "tool_calls"` means the model wants a tool, and says
nothing about whether the arguments are usable.

In [6]:
def lookup_inventory(sku: str) -> dict:
    """Stand-in for a real inventory service."""
    stock = {"A-100": 42, "B-200": 0}
    if sku.upper() not in stock:
        return {"sku": sku, "error": "unknown SKU"}
    return {"sku": sku, "quantity": stock[sku.upper()]}


TOOLS = [{
    "type": "function",
    "function": {
        "name": "lookup_inventory",
        "description": "Current stock level for a SKU.",
        "parameters": {
            "type": "object",
            "properties": {"sku": {"type": "string", "description": "SKU code, e.g. A-100"}},
            "required": ["sku"],
        },
    },
}]
IMPLEMENTATIONS = {"lookup_inventory": lookup_inventory}


def run_tool(name: str, args: dict) -> dict:
    """Refuse a call that names an unknown tool or does not match the tool's parameters."""
    spec = next((t["function"] for t in TOOLS if t["function"]["name"] == name), None)
    if spec is None:
        return {"error": f"unknown tool {name}"}
    params = spec["parameters"]
    missing = [k for k in params["required"] if k not in args]
    unknown = [k for k in args if k not in params["properties"]]
    wrong_type = [k for k in args if k in params["properties"] and params["properties"][k].get("type") == "string"
                  and not isinstance(args[k], str)]
    if missing or unknown or wrong_type:
        return {"error": f"bad arguments: missing {missing}, unknown {unknown}, not a string {wrong_type}"}
    return IMPLEMENTATIONS[name](**args)

In [7]:
messages = [{"role": "user", "content": "How many units of SKU A-100 do we have?"}]
for _ in range(5):  # bounded, so a model that keeps calling tools cannot loop forever
    response = client.chat.completions.create(
        model=GPT_OSS_120B, messages=messages, tools=TOOLS, max_tokens=1500
    )
    message = response.choices[0].message
    if not message.tool_calls:
        break
    messages.append(message.model_dump(exclude_none=True))
    for call in message.tool_calls:
        args = parse_json_lenient(call.function.arguments)
        result = run_tool(call.function.name, args)
        print(f"tool call: {call.function.name}({args}) -> {result}")
        messages.append({"role": "tool", "tool_call_id": call.id, "content": json.dumps(result)})
else:
    raise RuntimeError("no answer after 5 rounds of tool calls")

print("answer:", (message.content or "").strip())

tool call: lookup_inventory({'sku': 'A-100'}) -> {'sku': 'A-100', 'quantity': 42}


answer: We have **42 units** of SKU A‑100 in inventory.


## 6. Structured output

`response_format` with a JSON Schema asks the model for output in that shape. Check
`finish_reason` first, because a cut-off answer is cut-off JSON. Parse leniently,
because some models add text after a valid object. A schema controls the shape of the
output and has no say over the values in it, so the cell checks each value against the
source text.

In [8]:
SCHEMA = {
    "type": "object",
    "properties": {
        "invoice_id": {"type": "string"},
        "vendor": {"type": "string"},
        "total": {"type": "number"},
        "currency": {"type": "string"},
    },
    "required": ["invoice_id", "vendor", "total", "currency"],
    "additionalProperties": False,
}
SOURCE = "Invoice: INV-1042\nVendor: Acme Pty Ltd\nAmount due: AUD 1,280.50\nDue date: 30 October 2026"
EXPECTED = {"invoice_id": "INV-1042", "vendor": "Acme Pty Ltd", "total": 1280.5, "currency": "AUD"}

response = client.chat.completions.create(
    model=GPT_OSS_120B,
    messages=[{"role": "user", "content": (
        "Extract invoice_id, vendor, total and currency from this invoice as JSON. Write the "
        f"total as a number without a thousands separator, so 2,345.60 becomes 2345.6.\n\n{SOURCE}")}],
    response_format={"type": "json_schema", "json_schema": {"name": "invoice", "strict": True, "schema": SCHEMA}},
    max_tokens=1500,
)
choice = response.choices[0]
if choice.finish_reason != "stop":
    raise RuntimeError(f"incomplete output (finish_reason={choice.finish_reason}); raise max_tokens")

invoice = parse_json_lenient(choice.message.content)
print(json.dumps(invoice, indent=2))
wrong = {k: invoice.get(k) for k, v in EXPECTED.items() if invoice.get(k) != v}
print("values match the source" if not wrong else f"values that do NOT match the source: {wrong}")

{
  "invoice_id": "INV-1042",
  "vendor": "Acme Pty Ltd",
  "total": 1280.5,
  "currency": "AUD"
}
values match the source


## 7. Responses API and server-side state

gpt-oss also serves the Responses API on `bedrock-mantle`, at the same bare `/v1` path,
and that gives it server-side conversation state. With `store=True` the service keeps
the response, and the next request names it with `previous_response_id` instead of
resending the history. Stored responses are kept for 30 days in the Region that served
them
([`00-foundations/02`](../00-foundations/02-governance-projects-and-retention.ipynb)).
Chaining from a response created with `store=False` returns 404 `Response '…' not
found.` Unlike Chat Completions, the usage here itemises the reasoning tokens.

On `bedrock-runtime` the Responses API returns 404 for gpt-oss, so this state is a
`bedrock-mantle` feature. Keep structured output on Chat Completions: in September 2026
runs, `text.format` with a JSON Schema on this API returned nothing within 90 seconds on
either size.

In [9]:
first = client.responses.create(
    model=GPT_OSS_120B,
    input="My favourite colour is teal. Reply with one word: noted.",
    max_output_tokens=500,
    store=True,
)
second = client.responses.create(
    model=GPT_OSS_120B,
    input="What is my favourite colour? One word.",
    previous_response_id=first.id,  # the history comes from the server, not this request
    max_output_tokens=500,
    store=False,
)
details = second.usage.output_tokens_details if second.usage else None
print("turn 1:", first.output_text.strip())
print("turn 2:", second.output_text.strip())
print("output items:", [item.type for item in second.output])
print(f"status={second.status}  reasoning_tokens={getattr(details, 'reasoning_tokens', None)}")

client.responses.delete(first.id)
print("deleted the stored response")

turn 1: noted
turn 2: teal
output items: ['reasoning', 'message']
status=completed  reasoning_tokens=32
deleted the stored response


## 8. On `bedrock-runtime`: Converse

The runtime IDs carry the full `-1:0` suffix: `openai.gpt-oss-120b` on `bedrock-mantle`
is `openai.gpt-oss-120b-1:0` here, and both `openai.gpt-oss-120b-1` and the mantle ID
return `ValidationException: The provided model identifier is invalid.` Converse has one
request shape for every provider: `content` is a list of blocks, and settings such as
the token limit go in `inferenceConfig`. An option only one provider understands, such
as `reasoning_effort`, goes in `additionalModelRequestFields`, which Converse passes to
the model unchecked. Reasoning arrives as a `reasoningContent` block ahead of the text,
so read blocks by type instead of taking `content[0]`. The same IDs work with Chat
Completions at `bedrock-runtime`'s `/openai/v1` path, where the reasoning arrives inline
in `content`, as `<reasoning>…</reasoning>` ahead of the answer. The Converse tool loop
is in
[`00-foundations/04`](../00-foundations/04-bedrock-runtime-converse-and-profiles.ipynb).

In [10]:
response = runtime.converse(
    modelId="openai.gpt-oss-120b-1:0",
    messages=[{"role": "user", "content": [{"text": "What is the capital of Australia? Answer in one word."}]}],
    inferenceConfig={"maxTokens": 2000},
    additionalModelRequestFields={"reasoning_effort": "high"},
)
blocks = response["output"]["message"]["content"]
text = "".join(b["text"] for b in blocks if "text" in b)
print("blocks    :", [next(iter(b)) for b in blocks])
print("answer    :", text.strip())
trace = "".join(b["reasoningContent"].get("reasoningText", {}).get("text", "") for b in blocks if "reasoningContent" in b)
print("reasoning :", len(trace), "characters")
print("stopReason:", response["stopReason"])

blocks    : ['reasoningContent', 'text']
answer    : Canberra
reasoning : 203 characters
stopReason: end_turn


## Things that differ

What trips people up on this family, and what to do about it.

| Behaviour | What you see | What to do |
|---|---|---|
| Path prefix | `/openai/v1/chat/completions` returns 400 ``model `…` isn't supported on this route`` | Use the bare `/v1` path. `/openai/v1` is for the hosted OpenAI GPT, Gemma 4 and Grok models. |
| Empty answer | HTTP 200, `content` is `None`, `finish_reason` is `length` | The model spent `max_tokens` before answering, often on reasoning. Raise the budget and check `finish_reason` before using the text. |
| Structured output values | Valid JSON that holds a wrong value | A schema only controls the shape. Check the values you depend on. |
| Rate limits | 429 under load | Throttling is token-based and there is no requests-per-minute quota. Retry with backoff ([`00-foundations/03`](../00-foundations/03-scaling-tiers-and-latency.ipynb)). |
| Metrics | Nothing under `AWS/Bedrock` in CloudWatch | Mantle metrics are in the `AWS/BedrockMantle` namespace. |
| Conversation state | Chat Completions keeps none | Send the full `messages` list, or use the Responses API with `store=True` (section 7). |
| Structured output on the Responses API | `text.format` with a JSON Schema returned nothing within 90 seconds on both sizes (September 2026) | Use `response_format` on Chat Completions, with a client timeout (section 6). |
| JSON inside a code fence | `content` starts with a Markdown code fence although a strict schema was set (3 of 8 runs on 120B, September 2026) | Parse leniently; `parse_json_lenient` removes the fence. |
| Structured output on 20B | JSON that does not parse, starting with a doubled brace (5 of 8 runs, September 2026) | Use 120B for structured output, or validate and retry. |
| Forcing a tool call | `tool_choice` naming a function, or `required`, returns 200 and the model answers in text without calling the tool (4 of 4 runs, September 2026) | Leave `tool_choice` on `auto`, and check that a call came back before relying on one. |
| Runtime model IDs | `ValidationException: The provided model identifier is invalid.` for `openai.gpt-oss-120b` or `openai.gpt-oss-120b-1` | Send the full `-1:0` form, such as `openai.gpt-oss-120b-1:0` (section 8). |
| Responses API on `bedrock-runtime` | 404 `The model doesn't exist or doesn't support this API.` | Call the Responses API on `bedrock-mantle`, or use Converse or Chat Completions on `bedrock-runtime`. |
| Chat Completions on `bedrock-runtime` | The reasoning arrives inline in `content`, as `<reasoning>…</reasoning>` ahead of the answer | Strip the `<reasoning>` span before you use the text, or use Converse, which returns it as its own block. |

## Next

- Safeguard models, for classifying content against your own policy:
  [`02-gpt-oss-safeguard-classification`](02-gpt-oss-safeguard-classification.ipynb)
- OpenAI's hosted GPT models:
  [`01-openai-gpt/01`](../01-openai-gpt/01-responses-api-core.ipynb)
- Other families on Chat Completions:
  [`04-qwen/01`](../04-qwen/01-qwen3-core-and-tools.ipynb),
  [`05-deepseek/01`](../05-deepseek/01-deepseek-v3-reasoning.ipynb),
  [`08-moonshot-kimi/01`](../08-moonshot-kimi/01-kimi-k2.ipynb)
- Choosing a model and API across families:
  [`99-cross-cutting/01`](../99-cross-cutting/01-choosing-a-model-and-api.ipynb)
- Quotas, retries and latency:
  [`00-foundations/03`](../00-foundations/03-scaling-tiers-and-latency.ipynb)
- Production hardening:
  [`99-cross-cutting/03`](../99-cross-cutting/03-production-hardening.ipynb)